[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zoranduric/sd-hex-pipeline/blob/main/notebooks/build_balkans.ipynb)

# sd-hex-pipeline — build a Balkans hex map on Colab

Downloads commercially usable GIS data (Copernicus DEM, ESA WorldCover, Natural Earth, HydroRIVERS),
bakes it into an ~11 km pointy-top hex grid and writes `hexmap.sdhx` for Godot (`godot/hex_map_data.gd`).

CPU runtime is enough. **Runtime → Run all**; the result zip downloads at the end. See `DATA_SOURCES.md` for attribution you must ship with your game.

## 1. Settings

In [ ]:
REGION = "balkans"      # see `sdhex regions`, or set BBOX
BBOX = None             # e.g. "13,34.5,30,48.5"  (min_lon,min_lat,max_lon,max_lat)
HEX_KM = 11.0
USE_DRIVE = False       # True = cache downloads + outputs in Google Drive (asks for permission)

## 2. Install

In [ ]:
!pip -q install "git+https://github.com/zoranduric/sd-hex-pipeline"

## 3. Storage

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/sd-hex-pipeline"
else:
    BASE = "/content"
CACHE, OUT = f"{BASE}/cache", f"{BASE}/out"
os.makedirs(CACHE, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print(CACHE, OUT)

## 4. Build

In [ ]:
area = f'--bbox "{BBOX}"' if BBOX else f"--region {REGION}"
!sdhex build {area} --hex-km {HEX_KM} --out "{OUT}" --cache "{CACHE}" -v

## 5. Inspect

In [ ]:
import glob, json
from IPython.display import Image, display
run = sorted(glob.glob(f"{OUT}/*"), key=os.path.getmtime)[-1]
meta = json.load(open(f"{run}/hexmap.meta.json"))
print(run); print(json.dumps(meta.get("stats", meta.get("crop")), indent=1)[:1500])
display(Image(f"{run}/preview.png", width=900))

## 6. Download zip (skipped when using Drive)

In [ ]:
import shutil
if not USE_DRIVE:
    from google.colab import files
    zip_path = shutil.make_archive(f"/content/{os.path.basename(run)}", "zip", run)
    files.download(zip_path)
else:
    print("Saved to Drive:", run)
